In [3]:
import pandas as pd

In [4]:
clastry = pd.read_csv('dane/kontury_400m/warszawa_h3_400m_pelna.csv')

In [5]:
display(clastry.head())

,h3,population,population_2021,population_kontur,gestosc_os_km2,pow_km2,dzielnica,zrodlo,lon,lat
0,881f53d9ebfffff,218,199,857,332.0,0.6557,Białołęka,kontur,20.924063,52.349674
1,881f53d9e3fffff,75,69,401,165.0,0.4551,Białołęka,kontur,20.926367,52.342218
2,881f53d9ddfffff,314,287,831,504.0,0.6225,Białołęka,kontur,20.955437,52.364474
3,881f53d9d7fffff,554,507,1421,819.0,0.6766,Białołęka,kontur,20.970499,52.354494
4,881f53d9d5fffff,49,45,538,72.0,0.6766,Białołęka,kontur,20.957738,52.357019


In [6]:
wybrane = ['h3', 'population', 'gestosc_os_km2', 'pow_km2', 'dzielnica']

df = clastry[wybrane]

In [7]:
prognoza = pd.read_csv('dane/prognoza_ludnosci.csv')

In [8]:
display(prognoza.head())

,h3,dzielnica,pop_2025,pop_2026_niski,pop_2027_niski,pop_2028_niski,pop_2029_niski,pop_2030_niski,pop_2031_niski,pop_2032_niski,...,pop_2034_wysoki,pop_2035_wysoki,pop_2036_wysoki,pop_2036_wariant_gus,mieszkania_w_budowie,dryf_roczny_proc,dP_2031,dP_2036,dP_2031_norm,dP_2036_norm
0,881f53d9ebfffff,Białołęka,218,289,330,364,403,447,496,540,...,882,936,984,321,120,0.09,385,635,0.2905,0.3730
1,881f53d9e3fffff,Białołęka,75,75,75,75,75,75,75,75,...,76,76,76,75,0,0.09,0,1,0.0000,0.0148
2,881f53d9ddfffff,Białołęka,314,319,324,327,330,333,336,339,...,363,367,370,321,10,0.09,30,47,0.0811,0.1015
3,881f53d9d7fffff,Białołęka,554,562,565,585,627,651,656,661,...,742,748,753,580,61,0.09,140,165,0.1752,0.1901
4,881f53d9d5fffff,Białołęka,49,49,49,49,49,49,49,49,...,49,49,49,49,0,0.09,0,0,0.0000,0.0000


In [9]:
kolumny_bazowe = prognoza.columns[
    prognoza.columns.str.fullmatch(r"pop_\d{4}")
].tolist()

wybrane = ['h3', 'mieszkania_w_budowie', *kolumny_bazowe]

In [10]:
prognoza = prognoza[wybrane]

In [11]:
infrastruktura = pd.read_csv('dane/warszawa_infrastruktura.csv')

In [12]:
infrastruktura.head(5)

,kategoria,typ,nazwa,osm_id,geom_typ,pow_ha,dzielnica,h3,lon,lat
0,centrum_handlowe,mall,Centrum Handlowe Bemowo,210276753,obszar,0.05,Bemowo,881f523499fffff,20.928222,52.262789
1,centrum_handlowe,mall,Galeria Bemowo,22717599,obszar,1.83,Bemowo,881f523499fffff,20.930958,52.263967
2,centrum_handlowe,mall,Hala Wola,34988601,obszar,0.49,Bemowo,881f52265dfffff,20.914240,52.228380
3,centrum_handlowe,mall,Kaufland,549553102,obszar,0.33,Bemowo,881f522649fffff,20.895482,52.230985
4,centrum_handlowe,mall,Park Handlowy City Park Lazurowa,549553117,obszar,0.22,Bemowo,881f522649fffff,20.895516,52.229381


In [13]:
infrastruktura['typ'].value_counts()

typ
school                   708
christian                300
park                      77
mall                      75
theatre                   72
marketplace               59
hospital                  42
museum                    41
cinema                    29
town_hall                 18
christian_orthodox         8
christian_evangelical      4
buddhist                   4
christian_lutheran         3
muslim_sunni               2
christian_catholic         1
sikh                       1
christian_methodist        1
jewish                     1
Name: count, dtype: int64

In [14]:
mask = infrastruktura['typ'].isin([
    'christian', 'christian_orthodox', 'christian_evangelical',
    'buddhist', 'christian_lutheran', 'muslim_sunni',
    'christian_catholic', 'sikh', 'christian_methodist', 'jewish'
])

infrastruktura.loc[mask, 'typ'] = 'religious'

In [15]:
infrastruktura['typ'].value_counts()

typ
school         708
religious      325
park            77
mall            75
theatre         72
marketplace     59
hospital        42
museum          41
cinema          29
town_hall       18
Name: count, dtype: int64

In [16]:
tabela = pd.crosstab(
    infrastruktura['h3'],
    infrastruktura['typ']
)

tabela['liczba_mpp'] = tabela.sum(axis=1)

tabela

typ,cinema,hospital,mall,marketplace,museum,park,religious,school,theatre,town_hall,liczba_mpp
h3,,,,,,,,,,,
881f52240bfffff,0,0,0,0,0,0,0,2,0,0,2
881f522455fffff,0,0,0,0,0,0,1,0,0,0,1
881f522459fffff,0,0,0,0,0,0,0,1,0,0,1
881f52245dfffff,0,0,1,0,0,0,0,0,0,0,1
881f522481fffff,0,0,0,0,0,0,0,1,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...
881f53d9bbfffff,0,0,0,0,0,0,2,1,0,0,3
881f53d9bdfffff,0,0,0,0,0,0,1,1,0,0,2
881f53d9d3fffff,0,0,0,0,0,0,0,1,0,0,1


In [17]:
# Łączymy po kolumnie h3 (merge), a nie po numerze wiersza (concat):
# tabela z crosstab ma h3 w indeksie i tylko 408 klastrów z jakąkolwiek infrastrukturą.
tabela = tabela.reset_index()

wynik = (
    df.merge(prognoza, on="h3", how="left")
      .merge(tabela, on="h3", how="left")
)

# Klastry bez obiektów nie występują w tabeli -> 0 zamiast NaN.
kolumny_infra = tabela.columns.drop("h3")
wynik[kolumny_infra] = wynik[kolumny_infra].fillna(0).astype(int)

print(len(wynik), "wierszy, h3 unikalne:", wynik["h3"].is_unique)

848 wierszy, h3 unikalne: True


In [18]:
wynik.head()

,h3,population,gestosc_os_km2,pow_km2,dzielnica,mieszkania_w_budowie,pop_2025,pop_2026,pop_2027,pop_2028,...,hospital,mall,marketplace,museum,park,religious,school,theatre,town_hall,liczba_mpp
0,881f53d9ebfffff,218,332.0,0.6557,Białołęka,120,218,317,373,420,...,0,0,0,0,0,0,0,0,0,0
1,881f53d9e3fffff,75,165.0,0.4551,Białołęka,0,75,75,75,75,...,0,0,0,0,0,0,0,0,0,0
2,881f53d9ddfffff,314,504.0,0.6225,Białołęka,10,314,321,328,332,...,0,0,0,0,0,0,0,0,0,0
3,881f53d9d7fffff,554,819.0,0.6766,Białołęka,61,554,565,569,597,...,0,0,0,0,0,0,1,0,0,1
4,881f53d9d5fffff,49,72.0,0.6766,Białołęka,0,49,49,49,49,...,0,0,0,0,0,1,0,0,0,1


In [19]:
wynik.dtypes

h3                          str
population                int64
gestosc_os_km2          float64
pow_km2                 float64
dzielnica                   str
mieszkania_w_budowie      int64
pop_2025                  int64
pop_2026                  int64
pop_2027                  int64
pop_2028                  int64
pop_2029                  int64
pop_2030                  int64
pop_2031                  int64
pop_2032                  int64
pop_2033                  int64
pop_2034                  int64
pop_2035                  int64
pop_2036                  int64
cinema                    int64
hospital                  int64
mall                      int64
marketplace               int64
museum                    int64
park                      int64
religious                 int64
school                    int64
theatre                   int64
town_hall                 int64
liczba_mpp                int64
dtype: object

In [20]:
wynik.isna().sum()

h3                      0
population              0
gestosc_os_km2          0
pow_km2                 0
dzielnica               0
mieszkania_w_budowie    0
pop_2025                0
pop_2026                0
pop_2027                0
pop_2028                0
pop_2029                0
pop_2030                0
pop_2031                0
pop_2032                0
pop_2033                0
pop_2034                0
pop_2035                0
pop_2036                0
cinema                  0
hospital                0
mall                    0
marketplace             0
museum                  0
park                    0
religious               0
school                  0
theatre                 0
town_hall               0
liczba_mpp              0
dtype: int64